# 02 — Construction du dataset d'instruction-tuning (Phase 2)

Correspond à la Phase 2 de `docs/WORKFLOW.md`. Transforme `Train.csv` / `Val.csv` en exemples au format chat attendu par Gemma (`google/gemma-4-E2B-it`), avec rééquilibrage des langues sous-représentées, puis sauvegarde le résultat sur Drive pour la Phase 3 (fine-tuning).

## 1 — Récupérer le dépôt et charger les données

In [ ]:
import os

REPO_DIR = '/content/gemmafro-e2b'
if not os.path.isdir(REPO_DIR):
    !git clone https://github.com/andilMc/gemmafro-e2b.git {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

DATA_DIR = f'{REPO_DIR}/data'

In [ ]:
!pip install -q -U pandas scikit-learn datasets transformers sentencepiece

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT_DIR = '/content/drive/MyDrive/gemmafro-e2b'
PROCESSED_DIR = f'{PROJECT_DIR}/processed_data'
os.makedirs(PROCESSED_DIR, exist_ok=True)

In [ ]:
import pandas as pd

train = pd.read_csv(f'{DATA_DIR}/Train.csv')
val   = pd.read_csv(f'{DATA_DIR}/Val.csv')

print('Train:', train.shape, '| Val:', val.shape)

## 2 — Charger le tokenizer et définir le template de prompt

In [ ]:
from google.colab import userdata
from huggingface_hub import login
from transformers import AutoTokenizer

login(token=userdata.get('HF_TOKEN'))

MODEL_NAME = "google/gemma-4-E2B-it"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

In [ ]:
# subset = "<LangCode>_<CountryCode>" (ex. 'Amh_Eth') ; seul le préfixe identifie la langue.
SUBSET_TO_LANGUAGE = {
    'Eng': 'English',
    'Aka': 'Akan',
    'Lug': 'Luganda',
    'Swa': 'Swahili',
    'Amh': 'Amharic',
}

def subset_to_language_name(subset_code: str) -> str:
    if not subset_code or not isinstance(subset_code, str):
        return 'English'
    return SUBSET_TO_LANGUAGE.get(subset_code.split('_')[0], subset_code)

def build_prompt(question: str, language: str) -> str:
    return (
        f"Réponds à la question de santé suivante en {language}, "
        f"de façon claire et médicalement fiable.\n\nQuestion : {question}"
    )

## 3 — Mettre en forme les exemples au format chat

Deux colonnes par exemple : `prompt_text` (tour `user` seul, avec `add_generation_prompt=True`) et `full_text` (`user` + `model`). La Phase 3 masquera la perte sur les `len(tokenizer(prompt_text))` premiers tokens de `full_text` — cette approche par longueur mesurée est plus robuste qu'un pattern-matching sur les tokens spéciaux du template (qui peuvent différer d'une version de Gemma à l'autre) pour trouver où commence la réponse.

In [ ]:
def format_example(row):
    language = subset_to_language_name(row['subset'])
    prompt = build_prompt(row['input'], language)
    user_turn = [{"role": "user", "content": prompt}]
    full_turns = user_turn + [{"role": "model", "content": str(row['output'])}]
    return pd.Series({
        'prompt_text': tokenizer.apply_chat_template(user_turn, tokenize=False, add_generation_prompt=True),
        'full_text':   tokenizer.apply_chat_template(full_turns, tokenize=False),
    })

train[['prompt_text', 'full_text']] = train.apply(format_example, axis=1)
print(train.loc[0, 'full_text'])

## 4 — Split train/dev interne (stratifié par langue)
`Val.csv` reste intact comme jeu de validation final fidèle à l'évaluation Zindi ; ce split sert uniquement à surveiller l'overfitting pendant l'entraînement.

In [ ]:
from sklearn.model_selection import train_test_split

SEED = 42
train_split, dev_split = train_test_split(
    train, test_size=0.05, random_state=SEED, stratify=train['subset']
)
print(f'train_split : {len(train_split)} | dev_split : {len(dev_split)}')
train_split['subset'].value_counts()

## 5 — Rééquilibrage des langues sous-représentées

Sur-échantillonnage (avec remise) des langues minoritaires, plafonné à `MAX_OVERSAMPLE_FACTOR` fois leur taille d'origine — un rééquilibrage total sur la langue majoritaire (`Eng_Uga`, ~7 600 lignes) dupliquerait l'amharique (~1 800 lignes) plus de 4x, ce qui risque de faire mémoriser des réponses plutôt que généraliser. Le facteur plafond limite ce risque tout en réduisant le déséquilibre. Appliqué uniquement à `train_split` (jamais à `dev_split` ni `Val.csv`, qui doivent rester représentatifs de la vraie distribution).

In [ ]:
MAX_OVERSAMPLE_FACTOR = 3

majority_count = train_split['subset'].value_counts().max()

def rebalance(df, max_factor, majority_count, seed):
    parts = []
    for lang, group in df.groupby('subset'):
        target = min(majority_count, len(group) * max_factor)
        if target > len(group):
            extra = group.sample(target - len(group), replace=True, random_state=seed)
            group = pd.concat([group, extra], ignore_index=True)
        parts.append(group)
    return pd.concat(parts, ignore_index=True).sample(frac=1, random_state=seed).reset_index(drop=True)

train_balanced = rebalance(train_split, MAX_OVERSAMPLE_FACTOR, majority_count, SEED)

comparison = pd.DataFrame({
    'avant': train_split['subset'].value_counts(),
    'après': train_balanced['subset'].value_counts(),
}).sort_values('avant', ascending=False)
comparison

## 6 — Longueur des séquences formatées
Pour fixer `MAX_SEQ_LENGTH` en Phase 3 sans tronquer une trop grande part des exemples.

In [ ]:
def batch_token_lengths(texts, tokenizer, batch_size=256):
    lengths = []
    texts = [str(t) for t in texts]
    for i in range(0, len(texts), batch_size):
        enc = tokenizer(texts[i:i + batch_size], add_special_tokens=False)
        lengths.extend(len(ids) for ids in enc['input_ids'])
    return lengths

full_text_lengths = pd.Series(batch_token_lengths(train_balanced['full_text'], tokenizer))
print(full_text_lengths.describe(percentiles=[0.5, 0.9, 0.95, 0.99]))

MAX_SEQ_LENGTH = int(full_text_lengths.quantile(0.95))
print(f'\nMAX_SEQ_LENGTH suggéré (p95) : {MAX_SEQ_LENGTH}')

## 7 — Sauvegarder le dataset traité sur Drive
La Phase 3 charge directement ces fichiers plutôt que de refaire la mise en forme.

In [ ]:
# Val.csv : uniquement prompt_text (pour génération en Phase 4), pas de rééquilibrage ni de full_text.
val['prompt_text'] = val.apply(
    lambda row: tokenizer.apply_chat_template(
        [{"role": "user", "content": build_prompt(row['input'], subset_to_language_name(row['subset']))}],
        tokenize=False, add_generation_prompt=True,
    ),
    axis=1,
)

train_balanced[['ID', 'subset', 'prompt_text', 'full_text']].to_json(
    f'{PROCESSED_DIR}/train_balanced.jsonl', orient='records', lines=True, force_ascii=False
)
dev_split[['ID', 'subset', 'input', 'output', 'prompt_text', 'full_text']].to_json(
    f'{PROCESSED_DIR}/dev_split.jsonl', orient='records', lines=True, force_ascii=False
)
val[['ID', 'subset', 'input', 'output', 'prompt_text']].to_json(
    f'{PROCESSED_DIR}/val_formatted.jsonl', orient='records', lines=True, force_ascii=False
)

print('Sauvegardé dans', PROCESSED_DIR)
print(os.listdir(PROCESSED_DIR))

---
**Sorties de cette phase, à réutiliser en Phase 3 :**
- `train_balanced.jsonl` — jeu d'entraînement rééquilibré, prêt pour `SFTTrainer`
- `dev_split.jsonl` — 5 % interne pour surveiller l'overfitting
- `val_formatted.jsonl` — `Val.csv` avec prompts déjà formatés, pour l'évaluation en Phase 4
- `MAX_SEQ_LENGTH` suggéré (section 6) à reporter dans la config d'entraînement

**Étape suivante : Phase 3 — Fine-tuning (LoRA/QLoRA).**